In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy.signal import find_peaks
import math
import pandas as pd


# --- Load and preprocess the image ---
image_path = "../LongPathSPLA12.png" 
image = Image.open(image_path).convert("L")
#image = Image.open(image_path).convert("1")
coral = image
image_np = np.array(image)

In [ ]:
image_np

In [ ]:
unique, counts = np.unique(image_np, return_counts=True)
unique

In [ ]:
point_x, point_y = 13336, 1606

fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(image, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
ax.set_ylim(1700, 1200)
ax.set_xlim(12500,13600)

In [ ]:
point_x, point_y = 11723, 998

fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(image, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
ax.set_ylim(1700, 900)
ax.set_xlim(11300,13600)

def interpolate_pixels_along_line(x0, y0, x1, y1):
    """Uses Xiaolin Wu's line algorithm to interpolate all of the pixels along a
    straight line, given two points (x0, y0) and (x1, y1)

    Wikipedia article containing pseudo code that function was based off of:
        http://en.wikipedia.org/wiki/Xiaolin_Wu's_line_algorithm
    """
    pixels = []
    steep = abs(y1 - y0) > abs(x1 - x0)

    # Ensure that the path to be interpolated is shallow and from left to right
    if steep:
        t = x0
        x0 = y0
        y0 = t

        t = x1
        x1 = y1
        y1 = t

    if x0 > x1:
        t = x0
        x0 = x1
        x1 = t

        t = y0
        y0 = y1
        y1 = t

    dx = x1 - x0
    dy = y1 - y0
    gradient = dy / dx  # slope

    # Get the first given coordinate and add it to the return list
    x_end = round(x0)
    y_end = y0 + (gradient * (x_end - x0))
    xpxl0 = x_end
    ypxl0 = round(y_end)
    if steep:
        pixels.extend([(ypxl0, xpxl0), (ypxl0 + 1, xpxl0)])
    else:
        pixels.extend([(xpxl0, ypxl0), (xpxl0, ypxl0 + 1)])

    interpolated_y = y_end + gradient

    # Get the second given coordinate to give the main loop a range
    x_end = round(x1)
    y_end = y1 + (gradient * (x_end - x1))
    xpxl1 = x_end
    ypxl1 = round(y_end)

    # Loop between the first x coordinate and the second x coordinate, interpolating the y coordinates
    for x in range(xpxl0 + 1, xpxl1):
        if steep:
            pixels.extend([(math.floor(interpolated_y), x), (math.floor(interpolated_y) + 1, x)])

        else:
            pixels.extend([(x, math.floor(interpolated_y)), (x, math.floor(interpolated_y) + 1)])

        interpolated_y += gradient

    # Add the second given coordinate to the given list
    if steep:
        pixels.extend([(ypxl1, xpxl1), (ypxl1 + 1, xpxl1)])
    else:
        pixels.extend([(xpxl1, ypxl1), (xpxl1, ypxl1 + 1)])

    return pixels

pixels = interpolate_pixels_along_line(1606, 13336, 998, 11723)

res = np.array([image_np[pixels[i]] for i in range(len(pixels))])

plt.plot(res,'.')

import math

def interpolate_pixels_along_line(x0, y0, x1, y1):
    """Returns pixels along the line using Xiaolin Wu's algorithm (simplified)."""
    pixels = []
    steep = abs(y1 - y0) > abs(x1 - x0)

    # Swap coordinates if steep
    if steep:
        x0, y0 = y0, x0
        x1, y1 = y1, x1

    # Swap start/end so line goes left-to-right
    if x0 > x1:
        x0, x1 = x1, x0
        y0, y1 = y1, y0

    dx = x1 - x0
    dy = y1 - y0
    gradient = dy / dx if dx != 0 else 0

    # First endpoint
    x_end = round(x0)
    y_end = y0 + gradient * (x_end - x0)
    xpxl0, ypxl0 = x_end, round(y_end)
    pixels.append((ypxl0, xpxl0) if steep else (xpxl0, ypxl0))

    interpolated_y = y_end + gradient

    # Loop through main body
    x_end = round(x1)
    y_end = y1 + gradient * (x_end - x1)
    xpxl1, ypxl1 = x_end, round(y_end)

    for x in range(xpxl0 + 1, xpxl1):
        pixel = (math.floor(interpolated_y), x) if steep else (x, math.floor(interpolated_y))
        pixels.append(pixel)
        interpolated_y += gradient

    # Add last point
    pixels.append((ypxl1, xpxl1) if steep else (xpxl1, ypxl1))
    return pixels


def parallel_line_pixels(x0, y0, x1, y1, offset_distance=1, num_offsets=1):
    """
    Returns the main line pixels AND parallel pixels shifted perpendicular
    to the line by +/- offset_distance * k (for k=1..num_offsets).
    
    Args:
        x0, y0, x1, y1: endpoints of the original line
        offset_distance: distance in pixels between parallel lines
        num_offsets: how many offsets in each direction

    Returns:
        dict with:
            "main": pixels of the original line
            "parallel": list of lists, each list is a parallel offset line
    """
    # Original line pixels
    main_pixels = interpolate_pixels_along_line(x0, y0, x1, y1)

    # Direction vector of the line
    dx = x1 - x0
    dy = y1 - y0
    length = math.hypot(dx, dy)
    if length == 0:
        raise ValueError("Both points are the same.")

    # Unit perpendicular vector
    perp_x = -dy / length
    perp_y = dx / length

    all_parallel_lines = []

    for k in range(1, num_offsets + 1):
        shift = offset_distance * k

        # Shift in positive perpendicular direction
        shifted_pos = [
            (round(x + perp_x * shift), round(y + perp_y * shift))
            for (x, y) in main_pixels
        ]

        # Shift in negative perpendicular direction
        shifted_neg = [
            (round(x - perp_x * shift), round(y - perp_y * shift))
            for (x, y) in main_pixels
        ]

        all_parallel_lines.append(shifted_pos)
        all_parallel_lines.append(shifted_neg)

    return {
        "main": main_pixels,
        "parallel": all_parallel_lines
    }

# Example usage:
if __name__ == "__main__":
    x0, y0 = 10, 10
    x1, y1 = 50, 30
    result = parallel_line_pixels(x0, y0, x1, y1, offset_distance=2, num_offsets=2)

    print("Main line pixels:", result["main"][:10], "...")
    for idx, line in enumerate(result["parallel"]):
        print(f"Parallel line {idx+1}:", line[:10], "...")


parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=-2)

point_x, point_y = 11723, 998
par_pix = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(image, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(20):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
for i in range(10):
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_ylim(1700, 900)
ax.set_xlim(11300,13600)

print(len(par_pix))
print(len(pixels))

In [ ]:
import math

def interpolate_pixels_along_line(x0, y0, x1, y1):
    """
    Returns pixels along the line using Xiaolin Wu's algorithm (simplified).
    Produces a *continuous* set of points between (x0, y0) and (x1, y1).
    """
    pixels = []
    steep = abs(y1 - y0) > abs(x1 - x0)

    # Swap coordinates if steep
    if steep:
        x0, y0 = y0, x0
        x1, y1 = y1, x1

    # Swap start/end so line goes left-to-right
    if x0 > x1:
        x0, x1 = x1, x0
        y0, y1 = y1, y0

    dx = x1 - x0
    dy = y1 - y0
    gradient = dy / dx if dx != 0 else 0

    # First endpoint
    x_end = round(x0)
    y_end = y0 + gradient * (x_end - x0)
    xpxl0, ypxl0 = x_end, round(y_end)
    pixels.append((ypxl0, xpxl0) if steep else (xpxl0, ypxl0))

    interpolated_y = y_end + gradient

    # Loop through main body
    x_end = round(x1)
    y_end = y1 + gradient * (x_end - x1)
    xpxl1, ypxl1 = x_end, round(y_end)

    for x in range(xpxl0 + 1, xpxl1):
        pixel = (math.floor(interpolated_y), x) if steep else (x, math.floor(interpolated_y))
        pixels.append(pixel)
        interpolated_y += gradient

    # Add last point
    pixels.append((ypxl1, xpxl1) if steep else (xpxl1, ypxl1))
    return pixels


def offset_line_pixels(main_pixels, perp_x, perp_y, shift, keep_float=False):
    """
    Offset all pixels in a line by a perpendicular shift.
    
    Args:
        main_pixels: list of (x,y) points
        perp_x, perp_y: perpendicular unit vector
        shift: distance to shift
        keep_float: if True, keeps floating-point coords
    
    Returns:
        list of shifted pixels
    """
    shifted = []
    for (x, y) in main_pixels:
        new_x = x + perp_x * shift
        new_y = y + perp_y * shift
        if keep_float:
            shifted.append((new_x, new_y))
        else:
            # Use int() to consistently shift without collapsing too many points
            shifted.append((int(new_x), int(new_y)))
    return shifted


def parallel_line_pixels(x0, y0, x1, y1, offset_distance=1, num_offsets=1, keep_float=False):
    """
    Returns the main line pixels AND parallel pixels shifted perpendicular
    to the line by +/- offset_distance * k (for k=1..num_offsets).
    
    Args:
        x0, y0, x1, y1: endpoints of the original line
        offset_distance: distance in pixels between parallel lines
        num_offsets: how many offsets in each direction
        keep_float: if True, parallel lines keep floating coords instead of snapping
    
    Returns:
        dict with:
            "main": pixels of the original line
            "parallel": list of parallel lines (each is a list of pixels)
    """
    # Original line pixels
    main_pixels = interpolate_pixels_along_line(x0, y0, x1, y1)

    # Direction vector of the line
    dx = x1 - x0
    dy = y1 - y0
    length = math.hypot(dx, dy)
    if length == 0:
        raise ValueError("Both points are the same, cannot define a line.")

    # Unit perpendicular vector
    perp_x = -dy / length
    perp_y = dx / length

    all_parallel_lines = []

    for k in range(1, num_offsets + 1):
        shift = offset_distance * k

        # Shift in positive perpendicular direction
        shifted_pos = offset_line_pixels(main_pixels, perp_x, perp_y, shift, keep_float)

        # Shift in negative perpendicular direction
        shifted_neg = offset_line_pixels(main_pixels, perp_x, -perp_y, -shift, keep_float)

        all_parallel_lines.append(shifted_pos)
        all_parallel_lines.append(shifted_neg)

    return {
        "main": main_pixels,
        "parallel": all_parallel_lines
    }


# Example usage:
if __name__ == "__main__":
    x0, y0 = 10, 10
    x1, y1 = 50, 30
    
    result = parallel_line_pixels(
        x0, y0, x1, y1,
        offset_distance=2,   # 2 pixels apart
        num_offsets=2,       # 2 offsets in each direction
        keep_float=False     # snap to integer pixels
    )

    print("Main line pixels:", len(result["main"]))
    for idx, line in enumerate(result["parallel"]):
        print(f"Parallel line {idx+1} has {len(line)} pixels")


point_x, point_y = 11723, 998
pixels = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=20)['main']
par_pix = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(image, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(10):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
for i in range(10):
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_ylim(1700, 900)
ax.set_xlim(11300,13600)

res = np.array([image_np[pixels[i]] for i in range(len(pixels))])

In [ ]:
import numpy as np
import pandas as pd

def get_pixel_values_from_lines(image_np, lines):
    """
    Extracts pixel values from an image for a list of lines.
    Args:
        image_np: NumPy array of the image (H x W x C)
        lines: list of lists of (x, y) pixel coords
    Returns:
        List of 1D NumPy arrays, each containing values for one line.
    """
    line_values = []
    for line in lines:
        vals = np.array([image_np[y, x] for (x, y) in line])  # Note: (y,x) indexing
        line_values.append(vals)
    return line_values

def get_parallel_lines_dataframe(image_np, x0, y0, x1, y1, offset_distance=2, num_offsets=2):
    """
    Computes main + parallel lines and returns a DataFrame where:
      - 'main' = original line pixel values
      - 'p_1', 'p_2'... = positive offset lines
      - 'p_-1', 'p_-2'... = negative offset lines
    
    Works for grayscale or color images.
    """
    # Get all lines (main + parallels)
    lines_data = parallel_line_pixels(
        x0, y0, x1, y1,
        offset_distance=offset_distance,
        num_offsets=num_offsets,
        keep_float=False
    )
    
    # Combine main + parallels
    main_line = lines_data["main"]
    parallel_lines = lines_data["parallel"]
    
    all_lines = [main_line] + parallel_lines
    
    # Extract pixel values
    line_pixel_values = get_pixel_values_from_lines(image_np, all_lines)
    
    # Build column names:
    col_names = ["main"]
    # parallels are in order: +1, -1, +2, -2 ...
    for k in range(1, num_offsets + 1):
        col_names.append(f"p_{k}")    # positive
        col_names.append(f"p_{-k}")   # negative
    
    # Create DataFrame
    if image_np.ndim == 2:  # grayscale
        df = pd.DataFrame({col_names[i]: vals for i, vals in enumerate(line_pixel_values)})
    else:
        # For color images → store as tuples (R,G,B)
        df = pd.DataFrame({
            col_names[i]: [tuple(v) for v in vals]
            for i, vals in enumerate(line_pixel_values)
        })
    
    return df

# Example usage
if __name__ == "__main__":
    import cv2
    
    # Dummy RGB image (100x100)
    image = np.zeros((100, 100, 3), dtype=np.uint8)
    image[10:20, 10:20] = (255, 0, 0)  # red block
    
    # Define a line
    x0, y0 = 10, 10
    x1, y1 = 50, 30
    
    # Get DataFrame of main + 2 parallel lines
    df = get_parallel_lines_dataframe(
        image, x0, y0, x1, y1,
        offset_distance=2,
        num_offsets=2
    )
    
    print(df.head())
    print("Columns:", df.columns.tolist())


In [ ]:
df = get_parallel_lines_dataframe(image_np, 13336, 1606, 11723,998, offset_distance=10, num_offsets=10)
df_mean  = df.mean(axis=1)
df

In [ ]:
df_roll = df_mean.rolling(20, center=True).mean()

fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(df_mean)
ax.plot(df_roll)

In [ ]:
point_x, point_y = 11723, 998
pixels = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=20)['main']
par_pix = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(1606, 13336, 998, 11723, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(17):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_ylim(1700, 900)
ax.set_xlim(11250,13600)
plt.xticks(np.arange(11250,13600,250))
ax.grid()

In [ ]:
#df_mean.to_csv('profile1_250719.csv')

In [ ]:
point_x1, point_y1 = 13100, 1600
point_x2, point_y2 = 11723, 998
pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(13):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_ylim(1700, 700)
ax.set_xlim(10500,13600)
plt.xticks(np.arange(10500,13600,250))
ax.grid()

In [ ]:
df = get_parallel_lines_dataframe(image_np, point_x1, point_y1, point_x2, point_y2, offset_distance=10, num_offsets=10)
df_mean  = df.mean(axis=1)
df_roll = df_mean.rolling(20, center=True).mean()

fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(df_mean)
ax.plot(df_roll)

In [ ]:
point_x1, point_y1 = 12300, 1560
point_x2, point_y2 = 11000, 1230
pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(14):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_ylim(1700, 700)
ax.set_xlim(10500,13600)
plt.xticks(np.arange(10500,13600,250))
ax.grid()

In [ ]:
df = get_parallel_lines_dataframe(image_np, point_x1, point_y1, point_x2, point_y2, offset_distance=10, num_offsets=10)
df_mean  = df.mean(axis=1)
df_roll = df_mean.rolling(20, center=True).mean()

fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(df_mean)
ax.plot(df_roll)

In [ ]:
point_x1, point_y1 = 11750, 1420
point_x2, point_y2 = 9400, 830
pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(24):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_ylim(1700, 700)
ax.set_xlim(9000,12500)
plt.xticks(np.arange(9000,12500,250))
ax.grid()

In [ ]:
df = get_parallel_lines_dataframe(image_np, point_x1, point_y1, point_x2, point_y2, offset_distance=10, num_offsets=10)
df_mean  = df.mean(axis=1)
df_roll = df_mean.rolling(20, center=True).mean()

fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(df_mean)
ax.plot(df_roll)
ax.grid(axis='y')

In [ ]:
point_x1, point_y1 = 5848, 1470
point_x2, point_y2 = 3238, 880

pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(27):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_xlim(3000, 6000)
ax.set_ylim(1500,700)
#plt.xticks(np.arange(9000,12500,250))
ax.grid()

In [ ]:
df = get_parallel_lines_dataframe(image_np, point_x1, point_y1, point_x2, point_y2, offset_distance=10, num_offsets=10)
df_mean  = df.mean(axis=1)
df_roll = df_mean.rolling(20, center=True).mean()

fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(df_mean)
ax.plot(df_roll)
ax.grid(axis='y')

In [ ]:
point_x1, point_y1 = 3410, 1700
point_x2, point_y2 = 1114, 895

pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(23):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)
ax.set_xlim(1000, 3500)
ax.set_ylim(1800,800)
#plt.xticks(np.arange(9000,12500,250))
ax.grid()

In [ ]:
df = get_parallel_lines_dataframe(image_np, point_x1, point_y1, point_x2, point_y2, offset_distance=10, num_offsets=10)
df_mean  = df.mean(axis=1)
df_roll = df_mean.rolling(20, center=True).mean()

fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(df_mean)
ax.plot(df_roll)
ax.grid(axis='y')
ax.text(1000, 40,'this ones BW is reversed', bbox={'edgecolor':'k', 'facecolor':'w'})

In [ ]:
point_x1, point_y1 = 3410, 1700
point_x2, point_y2 = 1114, 895

pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

ax.imshow(coral, cmap='Greys_r')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(23):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)

line_x, line_y = [par_pix[420][1], par_pix2[420][1]], [par_pix[420][0], par_pix2[420][0]]

ax.plot(line_x, line_y, color='red', linewidth=3)
ax.set_xlim(1000, 3500)
ax.set_ylim(1800,800)
#plt.xticks(np.arange(9000,12500,250))
ax.grid()

In [ ]:
point_x1, point_y1 = 3410, 1700
point_x2, point_y2 = 1114, 895

pixels = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['main']
par_pix = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=20)['parallel'][0]
par_pix2 = parallel_line_pixels(point_y1, point_x1, point_y2, point_x2, offset_distance=-20)['parallel'][0]


fig, ax = plt.subplots(1, figsize=(12,5))

white = np.ones([coral.height, coral.width], dtype=np.uint8) * 255
ax.imshow(white, cmap='Greys')
ax.scatter(point_x, point_y, color='red', marker='o', s=10) # Plot point
for i in range(23):
    ax.scatter(pixels[::100][i][1],pixels[::100][i][0], color='blue', marker='o', s=10)
    ax.scatter(par_pix[::100][i][1],par_pix[::100][i][0], color='orange', marker='o', s=10)
    ax.scatter(par_pix2[::100][i][1],par_pix2[::100][i][0], color='red', marker='o', s=10)

line_x, line_y = [par_pix[420][1], par_pix2[420][1]], [par_pix[420][0], par_pix2[420][0]]

ax.plot(line_x, line_y, color='red', linewidth=3)
ax.set_xlim(1000, 3500)
ax.set_ylim(1800,800)
#plt.xticks(np.arange(9000,12500,250))
ax.grid()

In [ ]:
line = [par_pix[420][1], par_pix2[420][1]], [par_pix[420][0], par_pix2[420][0]]
line

In [ ]:
import math

def interpolate_pixels_along_line(x0, y0, x1, y1):
    """
    Returns pixels along the line using Xiaolin Wu's algorithm (simplified).
    Produces a *continuous* set of points between (x0, y0) and (x1, y1).
    """
    pixels = []
    steep = abs(y1 - y0) > abs(x1 - x0)

    # Swap coordinates if steep
    if steep:
        x0, y0 = y0, x0
        x1, y1 = y1, x1

    # Swap start/end so line goes left-to-right
    if x0 > x1:
        x0, x1 = x1, x0
        y0, y1 = y1, y0

    dx = x1 - x0
    dy = y1 - y0
    gradient = dy / dx if dx != 0 else 0

    # First endpoint
    x_end = round(x0)
    y_end = y0 + gradient * (x_end - x0)
    xpxl0, ypxl0 = x_end, round(y_end)
    pixels.append((ypxl0, xpxl0) if steep else (xpxl0, ypxl0))

    interpolated_y = y_end + gradient

    # Loop through main body
    x_end = round(x1)
    y_end = y1 + gradient * (x_end - x1)
    xpxl1, ypxl1 = x_end, round(y_end)

    for x in range(xpxl0 + 1, xpxl1):
        pixel = (math.floor(interpolated_y), x) if steep else (x, math.floor(interpolated_y))
        pixels.append(pixel)
        interpolated_y += gradient

    # Add last point
    pixels.append((ypxl1, xpxl1) if steep else (xpxl1, ypxl1))
    return pixels


def offset_line_pixels(main_pixels, perp_x, perp_y, shift, keep_float=False):
    """
    Offset all pixels in a line by a perpendicular shift.
    
    Args:
        main_pixels: list of (x,y) points
        perp_x, perp_y: perpendicular unit vector
        shift: distance to shift
        keep_float: if True, keeps floating-point coords
    
    Returns:
        list of shifted pixels
    """
    shifted = []
    for (x, y) in main_pixels:
        new_x = x + perp_x * shift
        new_y = y + perp_y * shift
        if keep_float:
            shifted.append((new_x, new_y))
        else:
            # Use int() to consistently shift without collapsing too many points
            shifted.append((int(new_x), int(new_y)))
    return shifted


def parallel_line_pixels(x0, y0, x1, y1, offset_distance=1, num_offsets=1, keep_float=False):
    """
    Returns the main line pixels AND parallel pixels shifted perpendicular
    to the line by +/- offset_distance * k (for k=1..num_offsets).
    
    Args:
        x0, y0, x1, y1: endpoints of the original line
        offset_distance: distance in pixels between parallel lines
        num_offsets: how many offsets in each direction
        keep_float: if True, parallel lines keep floating coords instead of snapping
    
    Returns:
        dict with:
            "main": pixels of the original line
            "parallel": list of parallel lines (each is a list of pixels)
    """
    # Original line pixels
    main_pixels = interpolate_pixels_along_line(x0, y0, x1, y1)

    # Direction vector of the line
    dx = x1 - x0
    dy = y1 - y0
    length = math.hypot(dx, dy)
    if length == 0:
        raise ValueError("Both points are the same, cannot define a line.")

    # Unit perpendicular vector
    perp_x = -dy / length
    perp_y = dx / length

    all_parallel_lines = []

    for k in range(1, num_offsets + 1):
        shift = offset_distance * k

        # Shift in positive perpendicular direction
        shifted_pos = offset_line_pixels(main_pixels, perp_x, perp_y, shift, keep_float)

        # Shift in negative perpendicular direction
        shifted_neg = offset_line_pixels(main_pixels, perp_x, perp_y, -shift, keep_float)

        all_parallel_lines.append(shifted_pos)
        all_parallel_lines.append(shifted_neg)

    return {
        "main": main_pixels,
        "parallel": all_parallel_lines
    }


# Example usage:
if __name__ == "__main__":
    x0, y0 = 10, 10
    x1, y1 = 50, 30
    
    result = parallel_line_pixels(
        x0, y0, x1, y1,
        offset_distance=2,   # 2 pixels apart
        num_offsets=2,       # 2 offsets in each direction
        keep_float=False     # snap to integer pixels
    )

    print("Main line pixels:", len(result["main"]))
    for idx, line in enumerate(result["parallel"]):
        print(f"Parallel line {idx+1} has {len(line)} pixels")


In [ ]:

x1 = 2
y1 = 1
x2 = 7
y2 = 18

#test = np.eye(10)
test = np.zeros(shape=(20,20))
mains = interpolate_pixels_along_line(x1,y1,x2,y2)
arr = np.array(mains).T
test[arr[0], arr[1]] = 1
fig, ax = plt.subplots(1, figsize=(4,4))
ax.imshow(test)

In [ ]:

x1 = 5
y1 = 1
x2 = 7
y2 = 18

#test = np.eye(10)
test = np.zeros(shape=(30,30))
res = parallel_line_pixels(y1,x1,y2,x2,2,1,keep_float=False)
main_ = res['main']
main_arr = np.array(main_).T
par_ = res['parallel'][0]
par_arr = np.array(par_).T
parx_ = res['parallel'][1]
parx_arr = np.array(parx_).T
test[main_arr[0], main_arr[1]] = 2
test[par_arr[0], par_arr[1]] = 1
#test[parx_arr[0], parx_arr[1]] = 3
fig, ax = plt.subplots(1, figsize=(5,5))
im = ax.imshow(test, cmap='jet', vmin=0, vmax=3)
plt.colorbar(im)

In [ ]:

x1 = 2
y1 = 1
x2 = 18
y2 = 10

#test = np.eye(10)
test = np.zeros(shape=(20,20))
res = parallel_line_pixels(y1,x1,y2,x2,2,1,keep_float=False)
main_ = res['main']
main_arr = np.array(main_).T
par_ = res['parallel'][0]
par_arr = np.array(par_).T
parx_ = res['parallel'][1]
parx_arr = np.array(parx_).T
test[main_arr[0], main_arr[1]] = 2
#test[par_arr[0], par_arr[1]] = 1
test[parx_arr[0], parx_arr[1]] = 3
fig, ax = plt.subplots(1, figsize=(5,5))
im = ax.imshow(test, cmap='jet', vmin=0, vmax=3)
plt.colorbar(im)

## Questions

- how shuold I use the index? X-Axis pixel counts are not that usefull for profile-image connection
- parallel pixel position accuracy. Steeper than 45° is problematic 
- 


In [ ]:
parallel_line_pixels()